In [ ]:
# hyperopt가 설치되어 있지 않은 경우 아래 코드를 실행하여 설치한다.
# 이미 설치되어 있으면 실행하지 않아도 된다.
#pip install hyperopt

In [ ]:
# hyperopt 라이브러리를 불러온다.
# HyperOpt는 베이지안 최적화 기반으로 하이퍼파라미터를 탐색할 때 사용하는 라이브러리이다.
import hyperopt

# 현재 설치된 hyperopt 라이브러리의 버전을 출력한다.
print(hyperopt.__version__)

In [ ]:
# hyperopt의 hp 모듈을 불러온다.
# hp는 하이퍼파라미터의 탐색 공간(search space)을 정의할 때 사용한다.
from hyperopt import hp

# -10 ~ 10까지 1간격을 가지는 입력 변수 x와 -15 ~ 15까지 1간격으로 입력 변수 y 설정.
# hp.quniform()은 지정한 범위 안에서 일정 간격을 갖는 값을 탐색하도록 설정하는 함수이다.
# 여기서는 x와 y라는 두 개의 입력 변수에 대해 탐색 공간을 정의한다.
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1) }

In [ ]:
# STATUS_OK는 목적 함수가 정상적으로 수행되었음을 표시할 때 사용하는 상태값이다.
from hyperopt import STATUS_OK

# 목적 함수를 생성. 변숫값과 변수 검색 공간을 가지는 딕셔너리를 인자로 받고, 특정 값을 반환
# HyperOpt는 이 목적 함수의 반환값이 최소가 되도록 입력값을 탐색한다.
def objective_func(search_space):
    # search_space 딕셔너리에서 x 값을 가져온다.
    x = search_space['x']

    # search_space 딕셔너리에서 y 값을 가져온다.
    y = search_space['y']

    # 최소화하고자 하는 목적 함수 값을 계산한다.
    # HyperOpt는 이 retval 값이 작아지는 방향으로 x, y 값을 탐색한다.
    retval = x**2 - 20*y

    # 계산된 목적 함수 값을 반환한다.
    return retval

In [ ]:
# fmin은 목적 함수의 최솟값을 찾기 위해 사용하는 HyperOpt의 핵심 함수이다.
# tpe는 Tree-structured Parzen Estimator 알고리즘으로, 베이지안 최적화 방식의 탐색 알고리즘이다.
# Trials는 탐색 과정에서 시도한 입력값과 결과값을 저장하는 객체이다.
from hyperopt import fmin, tpe, Trials

# 난수 생성 및 수치 계산을 위해 numpy를 불러온다.
import numpy as np

# 입력 결괏값을 저장한 Trials 객체값 생성.
# fmin()이 수행되는 동안 각 시도 결과가 trial_val에 저장된다.
trial_val = Trials()

# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_evals=5)로 찾아냄.
# fn에는 최소화할 목적 함수를 지정한다.
# space에는 앞에서 정의한 탐색 공간을 지정한다.
# algo=tpe.suggest는 TPE 알고리즘을 이용하여 다음 탐색 지점을 추천한다는 의미이다.
# max_evals=5는 총 5번만 탐색을 수행한다는 의미이다.
# rstate는 난수 시드를 고정하여 실행 결과가 재현되도록 한다.
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5
               , trials=trial_val, rstate=np.random.default_rng(seed=0))

# 5번 탐색했을 때 찾은 최적의 x, y 값을 출력한다.
print('best:', best_01)

In [ ]:
# 새로운 Trials 객체를 생성한다.
# 앞선 5회 탐색 결과와 분리하여 새롭게 탐색 결과를 저장하기 위함이다.
trial_val = Trials()

# max_evals를 20회로 늘려서 재테스트
# 탐색 횟수를 늘리면 더 좋은 최적값을 찾을 가능성이 커진다.
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20
               , trials=trial_val, rstate=np.random.default_rng(seed=0))

# 20번 탐색했을 때 찾은 최적의 x, y 값을 출력한다.
print('best:', best_02)

In [ ]:
# fmin( )에 인자로 들어가는 Trials 객체의 result 속성에 파이썬 리스트로 목적 함수 반환값들이 저장됨
# 리스트 내부의 개별 원소는 {'loss':함수 반환값, 'status':반환 상태값} 와 같은 딕셔너리임.
# 즉, 각 탐색 시도에서 목적 함수 값이 어떻게 나왔는지 확인할 수 있다.
print(trial_val.results)

In [ ]:
# Trials 객체의 vals 속성에 {'입력변수명':개별 수행 시마다 입력된 값 리스트} 형태로 저장됨.
# 즉, HyperOpt가 각 반복에서 선택한 x값과 y값의 목록을 확인할 수 있다.
print(trial_val.vals)

In [ ]:
# 탐색 결과를 DataFrame 형태로 정리하기 위해 pandas를 불러온다.
import pandas as pd

# results에서 loss 키값에 해당하는 밸류들을 추출하여 list로 생성.
# 각 탐색 시도에서 계산된 목적 함수 값을 losses 리스트에 저장한다.
losses = [loss_dict['loss'] for loss_dict in trial_val.results]

# DataFrame으로 생성.
# 각 반복에서 사용된 x, y 값과 그때의 손실값(losses)을 하나의 표로 정리한다.
result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})

# HyperOpt 탐색 결과를 DataFrame으로 확인한다.
result_df

### HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화

In [ ]:
# 아래 코드는 이전에 수록된 코드라 책에는 싣지 않았습니다.

# 데이터프레임 생성을 위해 pandas를 불러온다.
import pandas as pd

# 수치 계산 및 배열 처리를 위해 numpy를 불러온다.
import numpy as np

# 사이킷런에서 제공하는 위스콘신 유방암 데이터셋을 불러온다.
from sklearn.datasets import load_breast_cancer

# 학습 데이터와 테스트 데이터를 나누기 위해 train_test_split을 불러온다.
from sklearn.model_selection import train_test_split

# 경고 메시지를 숨기기 위해 warnings 모듈을 불러온다.
import warnings
warnings.filterwarnings('ignore')

# 위스콘신 유방암 데이터셋을 불러온다.
dataset = load_breast_cancer()

# 입력 변수 데이터를 DataFrame 형태로 변환한다.
# columns에는 각 feature의 이름을 사용한다.
cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)

# target 컬럼을 추가한다.
# target은 예측해야 하는 정답 레이블이다.
cancer_df['target']= dataset.target

# 마지막 target 컬럼을 제외한 나머지 컬럼을 입력 변수로 사용한다.
X_features = cancer_df.iloc[:, :-1]

# 마지막 target 컬럼을 정답 레이블로 사용한다.
y_label = cancer_df.iloc[:, -1]

In [ ]:
# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state를 설정하여 실행할 때마다 동일한 데이터 분할이 이루어지도록 한다.
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 앞에서 추출한 학습 데이터를 다시 학습과 검증 데이터로 분리
# 검증 데이터는 최종 모델 학습 시 조기 중단과 성능 확인에 사용된다.
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

In [ ]:
# hyperopt의 hp 모듈을 불러온다.
# XGBoost 하이퍼파라미터의 탐색 범위를 설정하기 위해 사용한다.
from hyperopt import hp

# max_depth는 5에서 20까지 1간격으로, min_child_weight는 1에서 2까지 1간격으로
# colsample_bytree는 0.5에서 1사이, learning_rate는 0.01에서 0.2 사이 정규 분포된 값으로 검색.
# max_depth는 트리의 최대 깊이를 의미한다.
# min_child_weight는 리프 노드가 되기 위한 최소 가중치 합을 의미하며, 과적합 제어에 사용된다.
# learning_rate는 각 트리가 전체 모델에 반영되는 비율이다.
# colsample_bytree는 트리를 만들 때 사용할 feature의 비율이다.
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
                   }

In [ ]:
# cross_val_score는 교차 검증을 통해 모델 성능을 평가할 때 사용한다.
from sklearn.model_selection import cross_val_score

# XGBoost의 사이킷런 래퍼 클래스인 XGBClassifier를 불러온다.
from xgboost import XGBClassifier

# STATUS_OK는 목적 함수가 정상적으로 수행되었음을 HyperOpt에 알려주는 상태값이다.
from hyperopt import STATUS_OK

# fmin()에서 입력된 search_space 값으로 입력된 모든 값은 실수형임.
# XGBClassifier의 정수형 하이퍼 파라미터는 정수형 변환을 해줘야 함.
# 정확도는 높을수록 더 좋은 수치임. -1 * 정확도를 곱해서 큰 정확도 값일수록 최소가 되도록 변환
def objective_func(search_space):
    # 수행 시간 절약을 위해 nestimators는 100으로 축소
    # HyperOpt가 선택한 하이퍼파라미터 조합을 이용하여 XGBoost 분류 모델을 생성한다.
    # max_depth와 min_child_weight는 정수형 파라미터이므로 int()로 변환한다.
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')

    # 생성된 XGBoost 모델을 3겹 교차 검증으로 평가한다.
    # scoring='accuracy'이므로 정확도를 기준으로 성능을 계산한다.
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

    # accuracy는 cv=3 개수만큼 roc-auc 결과를 리스트로 가짐. 이를 평균해서 반환하되 -1을 곱함.
    # HyperOpt의 fmin()은 loss가 최소가 되는 방향으로 탐색하기 때문에,
    # 정확도가 높을수록 loss가 작아지도록 -1을 곱해 반환한다.
    return {'loss':-1 * np.mean(accuracy), 'status': STATUS_OK}

In [ ]:
# fmin은 목적 함수가 최소가 되는 하이퍼파라미터 조합을 찾기 위한 함수이다.
# tpe는 HyperOpt에서 사용하는 베이지안 최적화 기반 탐색 알고리즘이다.
# Trials는 각 탐색 결과를 저장하는 객체이다.
from hyperopt import fmin, tpe, Trials

# HyperOpt 수행 결과를 저장할 Trials 객체를 생성한다.
trial_val = Trials()

# HyperOpt를 이용해 XGBoost 하이퍼파라미터 최적화를 수행한다.
# fn에는 최소화할 목적 함수를 입력한다.
# space에는 앞에서 정의한 XGBoost 하이퍼파라미터 탐색 공간을 입력한다.
# algo=tpe.suggest는 TPE 알고리즘으로 다음 하이퍼파라미터 조합을 선택한다는 의미이다.
# max_evals=50은 총 50번의 하이퍼파라미터 조합을 시도한다는 의미이다.
# trials에는 탐색 과정과 결과를 저장할 객체를 입력한다.
# rstate는 난수 시드를 고정하여 결과 재현성을 확보한다.
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수를 지정합니다.
            trials=trial_val, rstate=np.random.default_rng(seed=9))

# HyperOpt가 찾은 최적 하이퍼파라미터 조합을 출력한다.
print('best:', best)

In [ ]:
# HyperOpt가 찾은 최적 하이퍼파라미터 값을 보기 좋게 출력한다.
# colsample_bytree와 learning_rate는 실수형 값이므로 round()로 소수점 자릿수를 정리한다.
# max_depth와 min_child_weight는 정수형 파라미터이므로 int()로 변환하여 출력한다.
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

In [ ]:
# 오차 행렬과 정확도를 계산하기 위해 confusion_matrix, accuracy_score를 불러온다.
from sklearn.metrics import confusion_matrix, accuracy_score

# 정밀도와 재현율을 계산하기 위해 precision_score, recall_score를 불러온다.
from sklearn.metrics import precision_score, recall_score

# F1 score와 ROC-AUC를 계산하기 위해 f1_score, roc_auc_score를 불러온다.
from sklearn.metrics import f1_score, roc_auc_score

# 분류 모델의 평가 지표를 한 번에 출력하는 함수를 정의한다.
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차 행렬을 계산한다.
    # 실제값과 예측값을 비교하여 TN, FP, FN, TP를 확인할 수 있다.
    confusion = confusion_matrix( y_test, pred)

    # 정확도는 전체 데이터 중 올바르게 예측한 비율이다.
    accuracy = accuracy_score(y_test , pred)

    # 정밀도는 양성으로 예측한 것 중 실제 양성의 비율이다.
    precision = precision_score(y_test , pred)

    # 재현율은 실제 양성 중 모델이 양성으로 맞게 예측한 비율이다.
    recall = recall_score(y_test , pred)

    # F1 score는 정밀도와 재현율의 조화 평균이다.
    f1 = f1_score(y_test,pred)

    # ROC-AUC 추가
    # ROC-AUC는 모델이 양성과 음성을 얼마나 잘 구분하는지 나타내는 지표이다.
    roc_auc = roc_auc_score(y_test, pred_proba)

    # 오차 행렬을 출력한다.
    print('오차 행렬')
    print(confusion)

    # ROC-AUC print 추가
    # 정확도, 정밀도, 재현율, F1, AUC를 한 번에 출력한다.
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# HyperOpt가 찾은 최적 하이퍼파라미터를 적용하여 XGBClassifier 모델을 생성한다.
# n_estimators=400은 최대 400개의 부스팅 트리를 생성한다는 의미이다.
# learning_rate, max_depth, min_child_weight, colsample_bytree는 HyperOpt에서 찾은 값을 사용한다.
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5)
                           )

# 학습 과정에서 평가할 데이터셋을 지정한다.
# X_tr, y_tr은 학습 데이터이고 X_val, y_val은 검증 데이터이다.
evals = [(X_tr, y_tr), (X_val, y_val)]

# 최적 하이퍼파라미터가 적용된 XGBoost 모델을 학습한다.
# early_stopping_rounds=50은 검증 성능이 50번 반복 동안 개선되지 않으면 학습을 중단한다는 의미이다.
# eval_metric='logloss'는 평가 지표로 로그 손실을 사용한다는 의미이다.
# eval_set=evals는 학습 중 평가할 데이터셋 목록이다.
# verbose=True는 학습 과정을 출력하도록 한다.
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric='logloss',
                eval_set=evals, verbose=True)

# 학습된 XGBoost 모델을 이용하여 테스트 데이터의 클래스를 예측한다.
preds = xgb_wrapper.predict(X_test)

# 테스트 데이터가 클래스 1에 속할 예측 확률값을 추출한다.
# predict_proba 결과에서 두 번째 컬럼이 클래스 1에 대한 확률이다.
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

# 최적 하이퍼파라미터가 적용된 XGBoost 모델의 성능을 평가한다.
get_clf_eval(y_test, preds, pred_proba)